# Act 1 — Pub/Sub: the messaging backbone

Pub/Sub is GCP's flagship messaging primitive — a globally distributed publish-subscribe service that decouples producers from consumers. It's the substrate behind most event-driven architectures on GCP, and it's used internally by nearly every Google product that needs to fan out events at scale.

The shape is similar to Kafka or SNS+SQS, but the operational model is different: no brokers, no partitions to manage (in regular Pub/Sub), no zookeeper. You create topics, publish messages, and let GCP handle storage, replication, and delivery.

## The model — topics, subscriptions, messages

- **Topic** — a named channel. Producers publish messages to it.
- **Subscription** — a named consumer pipeline attached to a topic. Each subscription gets *its own copy* of every message published.
- **Message** — a blob of bytes plus a map of attributes (string key/value pairs), an optional ordering key, and a publish timestamp.

**One topic, multiple subscriptions** is the typical shape — same event delivered independently to N consumers (each tracks its own progress). Compare to Kafka where consumer groups serve this role; in Pub/Sub each subscription is the explicit configuration object.

**Two subscription delivery modes:**

- **Pull** — the subscriber calls Pub/Sub to fetch messages. Used by long-running workers (Dataflow, Cloud Run with a poller). The subscriber controls the rate.
- **Push** — Pub/Sub HTTP-POSTs each message to a URL you configure. Used to invoke Cloud Run / Cloud Functions automatically on each event. Pub/Sub controls the rate (with backoff on errors).

**At-least-once delivery is the default.** Messages may be redelivered if the subscriber's ack times out. Design consumers to be idempotent (notebook 13's CI patterns lean on this for retries).

**Exactly-once delivery** is opt-in per subscription (with some throughput and latency cost). Pub/Sub adds dedup tracking so a successfully-acked message is never redelivered. Worth turning on for any pipeline whose output is non-idempotent (charging a credit card, sending an email).

## Features that shape real designs

- **Ordering keys** — when set on publish, all messages sharing an ordering key are delivered to *that* subscription in publish order. Use for per-entity ordering (per-user, per-account). Without an ordering key, Pub/Sub doesn't guarantee order — the parallelism is the whole point.
- **Dead-letter topics** — after N delivery attempts to a subscription, route the message to a dead-letter topic. Standard pattern for handling poison messages without blocking the queue.
- **Schemas** — register an Avro or Protobuf schema with a topic; messages published with mismatched schemas are rejected. Useful for cross-team contracts.
- **Snapshots and seek** — `seek` lets a subscription replay from a past timestamp (within retention). `snapshot` captures a subscription's acked state for later restoration. Used for backfills and incident recovery.
- **Filtering** — subscriptions can carry an attribute filter (`attributes.event_type = "order_placed"`). Pub/Sub only delivers matching messages — saves you receiving-and-discarding on the consumer.
- **Retention** — messages are retained 7 days by default (configurable up to 31). Replay within that window via seek.

**Pub/Sub Lite** is a separate, lower-cost variant: regional (not global), partitioned (like Kafka), zonal storage. Used for very high-throughput workloads that can tolerate regional scope and want Kafka-like cost shape. Most teams should default to regular Pub/Sub; reach for Lite only with measured cost reasons.

# Act 2 — Eventarc, Workflows, Cloud Tasks, Cloud Scheduler

Pub/Sub is the messaging *substrate*. Built on top of it (and sometimes alongside it) are four products that shape *how* events become work: **Eventarc** for event routing, **Workflows** for orchestration, **Cloud Tasks** for rate-controlled async work, **Cloud Scheduler** for cron.

They overlap enough that picking the right one is the core decision; the act closes with a clear separation.

## Eventarc — event routing built on Pub/Sub

**Eventarc** is GCP's event-routing layer. You create a **trigger** that says "when event X happens, send it to destination Y." Behind the scenes Eventarc uses Pub/Sub topics and subscriptions; the abstraction layer adds source catalogues and standardised CloudEvents-formatted payloads.

**Event sources:**

- **Google sources** — GCS object events, Audit Log entries (admin actions, IAM changes), Cloud Build status changes, Firestore document writes, BigQuery job completion, ~140 services in total.
- **Custom sources** — publish a CloudEvent to a Pub/Sub topic; Eventarc routes from there.
- **Third-party** — supported event providers (e.g. SaaS apps that publish CloudEvents).

**Destinations:** Cloud Run services, Cloud Functions, Workflows, GKE workloads.

**Why use Eventarc over raw Pub/Sub:** the source catalogue. Wiring a GCS bucket to fire on object creation through Eventarc is one trigger config; through raw Pub/Sub you'd configure GCS notifications, the topic, the subscription, and the consumer separately. Eventarc is the easy button for "when *thing* happens in GCP, run my code."

## Workflows — orchestration as YAML

**Workflows** is GCP's orchestrator. You write a YAML definition that calls APIs (GCP, third-party, internal) in sequence, with control flow, parallel branches, retries, and error handling. The engine runs each step, retains state between calls, and handles long-running operations natively.

**Why this exists separately from Cloud Functions/Run:** a Workflow can wait for hours or days between steps (waiting on a human approval, a polling completion) without billing for idle compute. The state lives in the Workflow execution, not in a long-lived process. This makes it the GCP analogue of AWS Step Functions or Azure Logic Apps (Standard).

**Use Workflows for:**

- **Multi-step business processes** that span multiple services ("create order → call payment API → reserve inventory → notify shipping").
- **API orchestration** that needs retries and conditional branching.
- **Long-running async jobs** that wait on external completion.

**Don't use Workflows for:** high-frequency event handlers (a Cloud Function fits better — Workflows has overhead per execution), or anything where the business logic is more code than YAML (write a Cloud Run service instead).

## Cloud Tasks — rate-controlled task queue

**Cloud Tasks** is a task queue with **explicit rate control**. You create a queue, enqueue tasks (each task is an HTTP request to a URL you specify), and Cloud Tasks dispatches at a rate you configure.

**Key features:**

- **Rate limiting** — max dispatches per second, max concurrent dispatches, max retries with exponential backoff.
- **Scheduled tasks** — enqueue a task to run at a specific time (`schedule_time`). Used for delayed actions ("send a follow-up email in 24 hours").
- **HTTP targets** — tasks call Cloud Run, Cloud Functions, App Engine, or any HTTPS endpoint. The runtime is whatever serves the target URL.

**When to reach for Cloud Tasks over Pub/Sub:**

- You need rate control. Pub/Sub delivers as fast as the subscriber can ack; Cloud Tasks throttles. Use when downstream is rate-limited (third-party APIs, expensive operations).
- You need scheduled tasks at the task level (Pub/Sub doesn't have this — you'd use Cloud Scheduler + a topic).
- You're doing fan-out *to specific HTTP targets* and care about per-target rate limiting.

## Cloud Scheduler — managed cron

**Cloud Scheduler** is exactly what it sounds like — a fully-managed cron service. Schedules are crontab-format (`0 */4 * * *`). Targets are HTTP endpoints, Pub/Sub topics, or App Engine HTTP requests.

Use for: scheduled job triggers, periodic data syncs, daily report generation. Pair with Cloud Run Jobs (notebook 04) or Workflows for the actual work; Cloud Scheduler is just the trigger.

# Act 3 — Choose-what for messaging and orchestration

The overlap between Pub/Sub, Eventarc, Workflows, and Cloud Tasks confuses every team that's new to GCP. A short decision tree resolves most of the ambiguity.

## Decision tree

1. **One event, run one Cloud Run / Cloud Function?** → **Eventarc trigger.**
2. **One event, run N independent consumers, high throughput?** → **Pub/Sub topic with N subscriptions.**
3. **Multi-step business process spanning services?** → **Workflows.**
4. **Need explicit rate control or scheduled-per-task dispatch?** → **Cloud Tasks.**
5. **Periodic cron-style triggers?** → **Cloud Scheduler** (firing into one of the above).

**Common confusions resolved:**

- **"Should I use Eventarc or Pub/Sub directly?"** — Eventarc when the event source is a Google service Eventarc already knows about (GCS, Audit Logs, …). Direct Pub/Sub when you need fine-grained control over schema, ordering, or retention.
- **"Should I use Workflows or write a Cloud Run service?"** — Workflows when the orchestration is mostly API calls with branching/retries. Cloud Run when there's substantive business logic between calls.
- **"Should I use Cloud Tasks or Pub/Sub for retries?"** — Cloud Tasks if you need per-task retry config and rate control. Pub/Sub with dead-letter if you want broadcast semantics with retry-then-DLQ.

# Act 4 — API management: Apigee X vs API Gateway

When you publish APIs to external consumers (partners, mobile apps, third parties), you usually want a layer in front of your backends doing auth, rate limiting, transformation, analytics, and developer-portal hosting. GCP has two products at very different points on the complexity/cost curve.

## Apigee X vs API Gateway

**API Gateway** is the simpler managed gateway. You define an OpenAPI spec; API Gateway exposes it as an HTTPS endpoint and routes to a Cloud Run / Cloud Functions / App Engine backend. Built-in API key authentication, basic JWT validation. Pay-per-request, low fixed cost.

**Apigee X** is the full API management platform. Policy chains (transformations, fault handling, caching, OAuth flows), a developer portal for self-service onboarding, monetisation, deep analytics. Significant fixed cost. Used by enterprises whose APIs are products in their own right.

**Use API Gateway** for internal-facing APIs and simple external endpoints. **Use Apigee X** when you have an API program — versioned product APIs, dev portal, monetisation, complex transformations.

The choice is usually obvious from the cost/complexity profile. If you're asking the question, the answer is probably API Gateway.

## What carries into later chapters

Pub/Sub is the substrate for Cloud Audit Log streaming (notebook 12), Dataflow streaming pipelines (notebook 09), and most real-time integrations. Eventarc fires Cloud Run on GCS uploads, Audit Log changes, and Cloud Build status. Workflows orchestrates multi-step CI/CD steps when Cloud Build's linear model isn't enough (notebook 13).

Three habits to carry forward:

- **Pub/Sub is the default for async fan-out.** Cloud Tasks for rate-controlled HTTP fan-in. Don't mix the two up.
- **Idempotent consumers.** At-least-once delivery means messages can repeat; design for it.
- **Eventarc over raw Pub/Sub when wiring Google events.** Save yourself the topic/subscription plumbing.